# Practice Lab: CI/CD for AI Applications (Lesson 12.6)

Module 12 . 4 exercises . Prompt tests + canary + cost gate + full pipeline


# Lesson 12.6: CI/CD for AI Applications

4 exercises: 2E + 1M + 1C

All exercises run **locally** (pure Python simulations).


In [ ]:
import random
random.seed(42)


---
## Exercise 1 (Easy): 10-Test Regression Suite


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class TestSuite:
    def __init__(self,llm): self.llm=llm; self.results=[]
    def contains(self,name,prompt,phrases):
        out=self.llm(prompt).lower(); miss=[p for p in phrases if p.lower() not in out]
        self.results.append({"n":name,"ok":len(miss)==0,"d":miss or "OK"})
    def not_contains(self,name,prompt,forbidden):
        out=self.llm(prompt).lower(); found=[p for p in forbidden if p.lower() in out]
        self.results.append({"n":name,"ok":len(found)==0,"d":found or "OK"})
    def format_check(self,name,prompt,fn,desc=""):
        self.results.append({"n":name,"ok":fn(self.llm(prompt)),"d":desc})
    def similarity(self,name,prompt,ref,th=0.5):
        out=self.llm(prompt); ow=set(out.lower().split()); rw=set(ref.lower().split())
        sim=len(ow&rw)/max(len(rw),1)
        self.results.append({"n":name,"ok":sim>=th,"d":f"sim={sim:.2f}"})
    def report(self):
        p=sum(1 for r in self.results if r["ok"]); t=len(self.results)
        print(f"  Results: {p}/{t}")
        for r in self.results: print(f"    [{'PASS' if r['ok'] else 'FAIL'}] {r['n']:<28} {r['d']}")
        return p==t

def bot(p):
    p=p.lower()
    if "refund" in p: return "Refund: full within 7 days. 50% within 30 days. Contact support@netsetos.com."
    if "price" in p or "cost" in p: return '{"course":"GenAI","price":14999,"currency":"INR"}'
    if "schedule" in p or "class" in p: return "Live classes Mon-Fri 7:00-8:30 PM IST. Recordings available."
    if "prerequisite" in p: return "Prerequisites: Python basics, high school math."
    if "certificate" in p: return "Completion certificate after all modules and capstone."
    if "hello" in p or "hi" in p: return "Hello! I'm the Netsetos support assistant. How can I help?"
    return "I can help with course info, pricing, schedule, refunds. What do you need?"

s=TestSuite(bot)
print("10-Test Prompt Regression Suite:")
s.contains("refund_7days","Refund policy?",["7 days","refund"])
s.contains("refund_30days","Tell me about refunds",["30 days","50%"])
s.contains("refund_contact","How to get refund?",["support@netsetos.com"])
s.not_contains("no_competitors","Better than Coursera?",["coursera","udemy"])
s.not_contains("no_pii","Refund policy?",["credit card","ssn"])
s.format_check("price_json","Course price",lambda x: x.strip().startswith("{"),"JSON")
s.format_check("resp_length","Hello",lambda x: 10<len(x)<500,"10-500 chars")
s.similarity("greeting","Hi!","Hello! Netsetos support assistant. How can I help?")
s.similarity("schedule","When are classes?","Live classes Mon-Fri 7-8:30 PM IST recordings.")
s.contains("fallback","asdfghjkl random",["help","course"])
ok=s.report()
print(f"  Deploy: {'PROCEED' if ok else 'BLOCKED'}")


</details>


---
## Exercise 2 (Easy): Canary Simulation


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class Canary:
    def __init__(self,v1_err=0.01,v2_err=0.02): self.v1e=v1_err; self.v2e=v2_err
    def run(self,name,pct,n=1000):
        v1=v2=e1=e2=0
        for _ in range(n):
            if random.random()*100<pct:
                v2+=1; e2+=1 if random.random()<self.v2e else 0
            else:
                v1+=1; e1+=1 if random.random()<self.v1e else 0
        v2r=e2/max(v2,1)*100
        return {"name":name,"v1":v1,"v2":v2,"e1":e1,"e2":e2,"v2r":v2r}

c=Canary(0.01,0.02)
print("Canary Deployment:")
print(f"  {'Phase':<22} {'V1':>5} {'V2':>5} {'V1err':>6} {'V2err':>6} {'V2%':>6}")
for name,pct in [("Deploy 0%",0),("Canary 5%",5),("Canary 50%",50),("Full 100%",100)]:
    r=c.run(name,pct)
    print(f"  {r['name']:<22} {r['v1']:>5} {r['v2']:>5} {r['e1']:>6} {r['e2']:>6} {r['v2r']:>5.1f}%")
print(f"\n  Rollback trigger: v2 error > 5%")
print(f"  Rollback: gcloud run services update-traffic --to-revisions PREV=100")


</details>


---
## Exercise 3 (Medium): Cost Gate


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

prompts=["Course price?","Refund policy","Classes start?","Prerequisites?","Certificate?",
    "How long?","Pay EMI?","Instructor?","Module 11?","GenAI vs Python",
    "Hello","Tools learned?","Online?","Capstone?","Placement?",
    "What is RAG?","Agent architecture","Different from Scaler?","Schedule?","Demo class?"]

def sim_version(prompts,cost_1k):
    results=[]
    for p in prompts:
        ti=max(10,int(len(p.split())*1.3)); to=ti+random.randint(50,200)
        results.append({"tokens":ti+to,"cost":(ti+to)/1000*cost_1k})
    return results

curr=sim_version(prompts,0.00015)  # flash
new=sim_version(prompts,0.00125)   # pro

cc=sum(r["cost"] for r in curr); nc=sum(r["cost"] for r in new)
ct=sum(r["tokens"] for r in curr)/len(curr); nt=sum(r["tokens"] for r in new)/len(new)
inc=(nc-cc)/cc*100

print("Cost Gate:")
print(f"  Current (flash): ${cc:.6f} total, {ct:.0f} avg tokens")
print(f"  New (pro):       ${nc:.6f} total, {nt:.0f} avg tokens")
print(f"  Cost increase:   {inc:+.0f}%")
print(f"  Threshold:       20%")
print(f"  Decision:        {'BLOCKED' if inc>20 else 'APPROVED'}")
if inc>20: print(f"  Action: review with team lead. Options: keep flash, use routing, approve budget")


</details>


---
## Exercise 4 (Challenge): Full CI/CD Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
stages=[("Checkout",5,0,False),("Lint (ruff)",12,0,True),("Unit tests",25,0,True),
    ("Docker build",80,0.008,False),("Prompt regression",30,0,True),("Cost gate",20,0.003,True),
    ("Vuln scan (trivy)",15,0,True),("Push GCR",20,0.002,False),("Canary 5%",25,0.010,False),
    ("Promote 100%",10,0.005,False)]

print("Full AI CI/CD Pipeline:")
print(f"  {'#':>2} {'Stage':<22} {'Time':>6} {'Cost':>7} {'Status':>8}")
tt=tc=0
for i,(name,dur,cost,can_fail) in enumerate(stages):
    tt+=dur; tc+=cost
    m=dur//60; s=dur%60; ts=f"{m}m{s:02d}s" if m else f"{s:>3}s"
    print(f"  {i+1:>2} {name:<22} {ts:>6} ${cost:.3f}     PASS")
print(f"\n  Total: {tt//60}m{tt%60:02d}s | ${tc:.3f}")

# Fail scenario
fail_at=4; saved=sum(c for _,_,c,_ in stages[fail_at:])
print(f"\n  If prompt tests fail: skip stages 5-10, save ${saved:.3f}")
print(f"  Cloud Build: cloudbuild.yaml with secretManager")
print(f"  Slack: #deploys notification on success/failure")
print(f"  Rollback: traffic shift in ~15s, no rebuild")


</details>
